# CSRNet pipeline — smoke tests

This notebook checks that the full CSRNet pipeline works end-to-end **before running expensive training jobs**.
No GPU is required for any of these cells — everything runs on CPU.

**Sections**
1. Configuration — set your paths here
2. Imports — verify the whole module tree loads
3. Density map generation — shape check + visual
4. CSRNet architecture — forward pass shape + parameter count
5. Full prediction pipeline — fake image, then a real Qian Penguins sample
6. Evaluation functions — counting metrics + density bucket split
7. Training checklist — what to run on the cluster

## 1 · Configuration

Set `ROOT` to the repo root on your machine (or the cluster).

In [ ]:
import sys
from pathlib import Path

# ── change this to your repo root ──────────────────────────────────────────
ROOT = Path("/storage/homefs/as26q834/RobustAnimalCounting")   # cluster path
# ROOT = Path.cwd().parent                                      # if running from notebooks/
# ───────────────────────────────────────────────────────────────────────────

sys.path.insert(0, str(ROOT / "src"))

QIAN_ROOT   = ROOT / "data" / "splits" / "qian_penguins"
CSRNET_CKPT = ROOT / "results" / "csrnet" / "qian_penguins" / "best.pth"

print(f"Repo root : {ROOT}")
print(f"Qian data : {QIAN_ROOT}  (exists={QIAN_ROOT.exists()})")
print(f"Checkpoint: {CSRNET_CKPT}  (exists={CSRNET_CKPT.exists()})")

## 2 · Imports

If any of these fail, something is wrong with the installation or the `sys.path` above.

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image

from animal_counting.datasets.density_map import DensityMapDataset, generate_density_map
from animal_counting.datasets.qian_penguins import QianPenguinsDataset
from animal_counting.models.csrnet import CSRNet, CSRNetCountingModel
from animal_counting.evaluation import (
    count_metrics,
    split_by_density,
    evaluate_csrnet_density,
)

print("All imports OK ✓")
print(f"PyTorch version : {torch.__version__}")
print(f"CUDA available  : {torch.cuda.is_available()}")

## 3 · Density map generation

The key idea: each animal contributes a small Gaussian blob. Their sum equals the count. We test two things:
- The density map sum equals the number of points.
- After downsampling by 8× (what CSRNet does internally) the sum is preserved.

In [ ]:
import torch.nn.functional as F

# 5 fake animal positions inside a 512×512 image
points = np.array([[80, 100], [200, 150], [300, 300], [420, 80], [100, 400]], dtype=np.float32)

dm = generate_density_map(points, image_size=(512, 512), beta=0.3, k=3)

print(f"Number of points : {len(points)}")
print(f"Density map sum  : {dm.sum():.4f}  ← should be ~{len(points)}")

# Simulate the downsampling that DensityMapDataset applies
t = torch.from_numpy(dm).unsqueeze(0).unsqueeze(0)   # (1,1,512,512)
down = F.avg_pool2d(t, kernel_size=8, stride=8) * 64  # (1,1,64,64)
print(f"After 8× downsample sum: {down.sum().item():.4f}  ← should still be ~{len(points)}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Full-resolution density map
axes[0].imshow(dm, cmap="hot")
axes[0].scatter(points[:, 0], points[:, 1], c="cyan", s=40, label="GT points")
axes[0].set_title(f"Full-resolution density map (sum={dm.sum():.2f})")
axes[0].legend()
axes[0].axis("off")

# Downsampled (what the model sees as ground truth during training)
dm_down = down.squeeze().numpy()
axes[1].imshow(dm_down, cmap="hot")
axes[1].set_title(f"After 8× downsample (64×64, sum={dm_down.sum():.2f})")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 4 · CSRNet architecture

We check that the network's input→output shape is correct:
- Input: `(batch, 3, 512, 512)`
- Expected output: `(batch, 1, 64, 64)` — exactly 1/8 of the input on each side.

In [ ]:
net = CSRNet(pretrained=False)   # no internet needed
net.eval()

dummy = torch.zeros(1, 3, 512, 512)
with torch.no_grad():
    out = net(dummy)

print(f"Input shape  : {tuple(dummy.shape)}")
print(f"Output shape : {tuple(out.shape)}   ← expected (1, 1, 64, 64)")
assert out.shape == (1, 1, 64, 64), "Shape mismatch!"

total_params = sum(p.numel() for p in net.parameters())
trainable    = sum(p.numel() for p in net.parameters() if p.requires_grad)
print(f"\nTotal parameters    : {total_params:,}")
print(f"Trainable parameters: {trainable:,}")
print("Architecture check OK ✓")

## 5 · Full prediction pipeline

### 5a — Fake image (no data needed)

An untrained model will output a near-zero count — that's expected. What we're checking is that the whole `predict()` call runs without errors and returns the right types.

In [ ]:
model = CSRNetCountingModel(device="cpu", pretrained=False)

fake_image = Image.new("RGB", (512, 512), color=(120, 80, 60))
result = model.predict(fake_image)

print(f"Predicted count   : {result.count:.4f}  (untrained → near zero is fine)")
print(f"Density map shape : {tuple(result.density_map.shape)}   ← expected (1, 64, 64)")
assert result.density_map.shape == (1, 64, 64)
print("Prediction pipeline OK ✓")

### 5b — Real Qian Penguins sample

This requires `data/splits/qian_penguins/` to exist on disk. Run the cell — if the data isn't there yet it will print a warning and skip.

In [ ]:
if not QIAN_ROOT.exists():
    print(f"⚠️  Qian data not found at {QIAN_ROOT} — skipping real-data test.")
else:
    val_ds  = QianPenguinsDataset(root=QIAN_ROOT, split="val")
    dm_ds   = DensityMapDataset(val_ds, patch_size=512, augment=False)

    img_tensor, gt_density = dm_ds[0]
    gt_count = gt_density.sum().item()

    # Run the full-resolution image through the model (loaded from checkpoint if available)
    if CSRNET_CKPT.exists():
        model.load(CSRNET_CKPT)
        print(f"Loaded checkpoint from {CSRNET_CKPT}")
    else:
        print("No checkpoint found — using untrained weights (count will be wrong).")

    result = model.predict(img_tensor)  # accepts CHW tensor directly
    print(f"\nGround-truth count : {gt_count:.1f}")
    print(f"Predicted count    : {result.count:.1f}")

    # Visualize side by side
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))

    # Original image (de-normalize for display)
    mean = torch.tensor([0.485, 0.456, 0.406]).view(3,1,1)
    std  = torch.tensor([0.229, 0.224, 0.225]).view(3,1,1)
    img_display = (img_tensor * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    axes[0].imshow(img_display)
    axes[0].set_title(f"Image patch (GT count: {gt_count:.0f})")
    axes[0].axis("off")

    axes[1].imshow(gt_density.squeeze().numpy(), cmap="hot")
    axes[1].set_title("Ground-truth density map")
    axes[1].axis("off")

    axes[2].imshow(result.density_map.squeeze().numpy(), cmap="hot")
    axes[2].set_title(f"Predicted density map (count: {result.count:.1f})")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

## 6 · Evaluation functions

These come from the `evaluation` module (Valentin's work). We test them with dummy data so you can see exactly what they return — no real predictions needed.

In [ ]:
# Simulate predictions and ground-truth counts for 10 images
gt    = [3, 7, 15, 45, 80, 2, 60, 9, 30, 12]
pred  = [4, 6, 18, 40, 75, 3, 55, 8, 35, 10]

metrics = count_metrics(pred, gt)
print("count_metrics() output:")
for k, v in metrics.items():
    print(f"  {k:20s} {v}")

In [ ]:
# split_by_density bins images into sparse (≤10), medium (10-50), crowded (>50)
image_ids = [f"img_{i:02d}" for i in range(len(gt))]
buckets = split_by_density(image_ids, pred, gt)

print("Density bucket split:")
for name, group in buckets.items():
    n = len(group["gt_counts"])
    print(f"  {name:8s} → {n} images  gt={group['gt_counts']}")

In [ ]:
# evaluate_csrnet_density computes MAE/RMSE/relative_error per bucket + SSIM
# For the SSIM part we pass empty lists (no density maps in this dry run)
results = evaluate_csrnet_density(
    image_ids=image_ids,
    pred_counts=pred,
    gt_counts=gt,
    pred_maps=[],   # will give SSIM=NaN — that's fine for this test
    gt_maps=[],
)

print("evaluate_csrnet_density() results:")
for bucket, m in results.items():
    print(f"  [{bucket}]")
    for k, v in m.items():
        print(f"    {k:20s} {v}")

## 7 · Training checklist (for Valentin — GPU required)

Everything above passed? Then training is ready to go. Run these commands **from the repo root** on the cluster.

### Submit the jobs

```bash
# CSRNet on Qian Penguins (dense scenes — tests H2)
sbatch sbatch_scripts/train_csrnet_qian.sh

# YOLOv8 on Eikelboom (sparse scenes — tests H1)
sbatch sbatch_scripts/train_yolov8_eikelboom.sh

# YOLOv8 on Delplanque (for cross-domain — tests H4)
sbatch sbatch_scripts/train_yolov8_delplanque.sh
```

### Monitor progress

```bash
squeue -u $USER                                           # see running jobs
tail -f sbatch_scripts/logs/<job_name>_<job_id>.logs      # live training output
```

CSRNet prints `val_MAE` and `val_RMSE` every epoch. It saves `results/csrnet/qian_penguins/best.pth` whenever the MAE improves, and stops automatically after 20 epochs without improvement.

### After training — run evaluation

```bash
# YOLOv8 in-domain (Eikelboom trained, Eikelboom tested, per density bucket)
PYTHONPATH=src python scripts/eval/yolov8/evaluate.py \
    --train-dataset eikelboom \
    --test-dataset eikelboom \
    --weights results/yolov8/eikelboom/weights/best.pt \
    --mode density

# YOLOv8 cross-domain (trained on Eikelboom, tested on Delplanque)
PYTHONPATH=src python scripts/eval/yolov8/evaluate.py \
    --train-dataset eikelboom \
    --test-dataset delplanque \
    --weights results/yolov8/eikelboom/weights/best.pt \
    --mode cross
```

Results are saved as JSON in `results/yolov8/`.